# 06 — Dual-Branch Dataset Class (Physics-Informed GAT, Phase 7)

This is **Phase 7** of the diabetic foot thermography project. Inputs are `data/processed/split_manifest.csv` (334 rows, one per `subject_id` + `foot_side`, produced by `04_patient_split.ipynb`) and `data/processed/graph_manifest.csv` (334 rows, produced by `05_angiosome_graph.ipynb`). This notebook does **not** re-parse the raw Excel file, does **not** re-walk the `ThermoDataBase` directory tree, and does **not** recompute anything from Phases 1–6 that is already saved to disk — it only reads the `.npy`, `.png`, and `.npz` artifacts those phases already produced.

**Why no PyTorch Geometric.** Every graph in this dataset has **exactly 4 nodes** and either 8 or 12 directed edges depending on topology variant — identical structure for all 334 subject+foot graphs; only node/edge *features* vary, never the shape. This is unusually regular for graph data. Because size never varies, graphs can be represented as small dense tensors and batched with PyTorch's default `collate_fn` with zero custom batching logic — a sparse graph library is unnecessary overhead here. Phase 8's GAT layer will implement attention over these 4 fixed nodes directly in plain PyTorch, so graphs are handled as plain fixed-size tensors below, and images are loaded directly via PIL/NumPy rather than `torchvision`.

**What this notebook builds:**

1. A merged, cross-validated `combined_manifest_df` (`split_manifest_df` + `graph_manifest_df`).
2. Train-only normalization statistics (node features + raw/relative tissue pixels), saved to `normalization_stats.json` as the single source of truth for every later phase.
3. `PlantarThermographyDataset`, a `torch.utils.data.Dataset` returning the dual-branch (graph + image) sample dict Phase 8's model will consume.
4. `train_loader` / `val_loader` / `test_loader`, plus batch- and label-level sanity checks.

`RANDOM_SEED = 42` throughout, matching the convention established in `02_eda.ipynb` through `05_angiosome_graph.ipynb`.


## Section 1 — Setup

Load `split_manifest.csv` and `graph_manifest.csv` and merge them on `['subject_id', 'foot_side']` into one `combined_manifest_df`. Both source files independently carry `group`, `gender`, and `split` columns (the former from Phase 5, the latter from Phase 6) — before trusting the merge, every row's values for these three columns are checked for exact agreement between the two source files. Any mismatch would mean the two manifests drifted out of sync between Phase 5 and Phase 6, so this check runs before anything downstream depends on the merged result.


In [2]:
print("=== SECTION 1: Setup ===")

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Running log of any issues found, printed in the final summary cell
issues_log = []

# --- Inputs (already-processed; do not re-derive from raw Excel or the raw directory tree) ---
PROCESSED_DIR = Path("data/processed")
SPLIT_MANIFEST_PATH = PROCESSED_DIR / "split_manifest.csv"
GRAPH_MANIFEST_PATH = PROCESSED_DIR / "graph_manifest.csv"

# --- Output (single source of truth for normalization in every later phase) ---
NORM_STATS_PATH = PROCESSED_DIR / "normalization_stats.json"

split_manifest_df = pd.read_csv(SPLIT_MANIFEST_PATH)
graph_manifest_df = pd.read_csv(GRAPH_MANIFEST_PATH)

print(f"split_manifest_df shape: {split_manifest_df.shape} (expected (334, 15))")
print(split_manifest_df.dtypes)
print()
print(f"graph_manifest_df shape: {graph_manifest_df.shape} (expected (334, 9))")
print(graph_manifest_df.dtypes)

if split_manifest_df.shape[0] != 334:
    issue = f"SECTION 1: expected 334 rows in split_manifest_df, found {split_manifest_df.shape[0]}"
    print(f"!!! {issue}")
    issues_log.append(issue)
if graph_manifest_df.shape[0] != 334:
    issue = f"SECTION 1: expected 334 rows in graph_manifest_df, found {graph_manifest_df.shape[0]}"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 1: Setup ===
split_manifest_df shape: (334, 15) (expected (334, 15))
subject_id               str
group                    str
gender                   str
foot_side                str
original_height        int64
original_width         int64
pad_row_offset         int64
pad_col_offset         int64
canonical_height       int64
canonical_width        int64
raw_array_path           str
mask_path                str
relative_array_path      str
colormap_png_path        str
split                    str
dtype: object

graph_manifest_df shape: (334, 9) (expected (334, 9))
subject_id          str
foot_side           str
group               str
gender              str
split               str
npz_path            str
num_nodes         int64
num_edges_base    int64
num_edges_diag    int64
dtype: object


In [3]:
print("=== SECTION 1 (cont.): Merge & Agreement Check ===")

SHARED_COLS = ["group", "gender", "split"]

# Merge on the join key only, then compare the shared columns from each side explicitly
# (pandas would otherwise silently suffix them as _x/_y and hide any drift).
merged = split_manifest_df.merge(
    graph_manifest_df,
    on=["subject_id", "foot_side"],
    suffixes=("_split", "_graph"),
    how="inner",
)

mismatch_mask = pd.Series(False, index=merged.index)
mismatch_details = {}
for col in SHARED_COLS:
    col_mismatch = merged[f"{col}_split"] != merged[f"{col}_graph"]
    mismatch_details[col] = int(col_mismatch.sum())
    mismatch_mask |= col_mismatch

print("Agreement check (split_manifest.csv vs. graph_manifest.csv) for shared columns:")
for col, n_bad in mismatch_details.items():
    status = "PASS" if n_bad == 0 else "FAIL"
    print(f"  {col}: {n_bad} mismatches -- {status}")

n_total_mismatches = int(mismatch_mask.sum())
if n_total_mismatches > 0:
    issue = f"SECTION 1: {n_total_mismatches} rows have group/gender/split disagreement between the two manifests"
    print(f"!!! {issue}")
    issues_log.append(issue)
    print(merged.loc[mismatch_mask, ["subject_id", "foot_side"] + [f"{c}_split" for c in SHARED_COLS] + [f"{c}_graph" for c in SHARED_COLS]])
else:
    print("OK -- group/gender/split agree exactly on every row across both manifests.")

# Build the single combined_manifest_df: keep one copy of each shared column (drop the _graph
# duplicates), plus every other column each source file uniquely contributes.
combined_manifest_df = merged.drop(columns=[f"{c}_graph" for c in SHARED_COLS])
combined_manifest_df = combined_manifest_df.rename(columns={f"{c}_split": c for c in SHARED_COLS})

print()
print(f"combined_manifest_df shape: {combined_manifest_df.shape} (expected (334, 19))")
if combined_manifest_df.shape[0] != 334:
    issue = f"SECTION 1: combined_manifest_df has {combined_manifest_df.shape[0]} rows, expected 334"
    print(f"!!! {issue}")
    issues_log.append(issue)

print()
print("Value counts of the 'split' column:")
print(combined_manifest_df["split"].value_counts())
print()
print(combined_manifest_df.columns.tolist())


=== SECTION 1 (cont.): Merge & Agreement Check ===
Agreement check (split_manifest.csv vs. graph_manifest.csv) for shared columns:
  group: 0 mismatches -- PASS
  gender: 0 mismatches -- PASS
  split: 0 mismatches -- PASS
OK -- group/gender/split agree exactly on every row across both manifests.

combined_manifest_df shape: (334, 19) (expected (334, 19))

Value counts of the 'split' column:
split
train    232
test      52
val       50
Name: count, dtype: int64

['subject_id', 'group', 'gender', 'foot_side', 'original_height', 'original_width', 'pad_row_offset', 'pad_col_offset', 'canonical_height', 'canonical_width', 'raw_array_path', 'mask_path', 'relative_array_path', 'colormap_png_path', 'split', 'npz_path', 'num_nodes', 'num_edges_base', 'num_edges_diag']


## Section 2 — Normalization Stats (Computed From Train Split Only)

**Critical for avoiding data leakage: every statistic below is computed from `split == "train"` rows only, never from val or test.**

- **(a) Node features:** for every train-split graph's `.npz`, load `node_features` `(4, 6)`, stack across all train graphs (`num_train_graphs x 4` rows total), and compute the per-column mean/std.
- **(b) Image pixel stats:** for every train-split subject+foot, load the raw array and its tissue mask, and compute one global mean/std over *only* the `True` (tissue) pixels — never the padded background — accumulated across the whole train split. Repeated separately for the relative array.
- **(c)** Both are saved to `data/processed/normalization_stats.json`, which becomes the single source of truth for normalization in every later phase — no later notebook should recompute these statistics from raw data again.


In [4]:
print("=== SECTION 2a: Normalization Stats -- Node Features (train split only) ===")

NODE_FEATURE_NAMES = [
    "mean_abs_temp", "std_abs_temp", "mean_rel_temp",
    "pixel_count", "centroid_row_norm", "centroid_col_norm",
]

train_df = combined_manifest_df.loc[combined_manifest_df["split"] == "train"].reset_index(drop=True)
print(f"Train-split rows: {len(train_df)}")

all_train_node_features = []
for _, row in train_df.iterrows():
    with np.load(row["npz_path"]) as npz:
        nf = npz["node_features"]  # (4, 6)
    if nf.shape != (4, 6):
        issue = f"SECTION 2a: {row['subject_id']}_{row['foot_side']} node_features shape {nf.shape} != (4, 6)"
        print(f"!!! {issue}")
        issues_log.append(issue)
        continue
    all_train_node_features.append(nf)

all_train_node_features = np.concatenate(all_train_node_features, axis=0)  # (num_train_graphs*4, 6)
print(f"Stacked train node feature matrix shape: {all_train_node_features.shape} "
      f"(expected ({len(train_df)} x 4, 6) = ({len(train_df) * 4}, 6))")
if all_train_node_features.shape[0] != len(train_df) * 4:
    issue = (f"SECTION 2a: stacked node feature row count {all_train_node_features.shape[0]} "
              f"!= expected {len(train_df) * 4}")
    print(f"!!! {issue}")
    issues_log.append(issue)

node_feature_mean = all_train_node_features.mean(axis=0)
node_feature_std = all_train_node_features.std(axis=0)

print()
print("Per-feature mean / std across all training nodes:")
for name, m, s in zip(NODE_FEATURE_NAMES, node_feature_mean, node_feature_std):
    print(f"  {name:20s}  mean={m: .6f}  std={s: .6f}")


=== SECTION 2a: Normalization Stats -- Node Features (train split only) ===
Train-split rows: 232
Stacked train node feature matrix shape: (928, 6) (expected (232 x 4, 6) = (928, 6))

Per-feature mean / std across all training nodes:
  mean_abs_temp         mean= 28.866394  std= 2.974192
  std_abs_temp          mean= 0.941301  std= 0.426610
  mean_rel_temp         mean=-0.032554  std= 0.669784
  pixel_count           mean= 1935.525879  std= 1209.862549
  centroid_row_norm     mean= 0.545102  std= 0.240834
  centroid_col_norm     mean= 0.502554  std= 0.209347


In [5]:
print("=== SECTION 2b: Normalization Stats -- Raw/Relative Image Pixel Stats (train split only) ===")

raw_tissue_values = []
relative_tissue_values = []

for _, row in train_df.iterrows():
    mask = np.load(row["mask_path"])          # boolean, (208, 96)
    raw_arr = np.load(row["raw_array_path"])          # float32, (208, 96)
    rel_arr = np.load(row["relative_array_path"])      # float32, (208, 96)

    raw_tissue_values.append(raw_arr[mask])
    relative_tissue_values.append(rel_arr[mask])

raw_tissue_values = np.concatenate(raw_tissue_values)
relative_tissue_values = np.concatenate(relative_tissue_values)

raw_tissue_mean = float(raw_tissue_values.mean())
raw_tissue_std = float(raw_tissue_values.std())
relative_tissue_mean = float(relative_tissue_values.mean())
relative_tissue_std = float(relative_tissue_values.std())

print(f"Total train tissue pixels pooled: {len(raw_tissue_values)} (raw), {len(relative_tissue_values)} (relative)")
print()
print(f"raw_tissue_mean:      {raw_tissue_mean:.6f}")
print(f"raw_tissue_std:       {raw_tissue_std:.6f}")
print(f"relative_tissue_mean: {relative_tissue_mean:.6f}")
print(f"relative_tissue_std:  {relative_tissue_std:.6f}")


=== SECTION 2b: Normalization Stats -- Raw/Relative Image Pixel Stats (train split only) ===
Total train tissue pixels pooled: 1634902 (raw), 1634902 (relative)

raw_tissue_mean:      28.908850
raw_tissue_std:       3.137550
relative_tissue_mean: -0.000000
relative_tissue_std:  1.245347


In [6]:
print("=== SECTION 2c: Save normalization_stats.json ===")

normalization_stats = {
    "node_feature_names": NODE_FEATURE_NAMES,
    "node_feature_mean": node_feature_mean.tolist(),
    "node_feature_std": node_feature_std.tolist(),
    "raw_tissue_mean": raw_tissue_mean,
    "raw_tissue_std": raw_tissue_std,
    "relative_tissue_mean": relative_tissue_mean,
    "relative_tissue_std": relative_tissue_std,
    "computed_from_split": "train",
    "train_subject_foot_count": int(len(train_df)),
}

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
with open(NORM_STATS_PATH, "w") as f:
    json.dump(normalization_stats, f, indent=2)

print(f"Saved: {NORM_STATS_PATH}")
print()
print(json.dumps(normalization_stats, indent=2))

expected_keys = {
    "node_feature_names", "node_feature_mean", "node_feature_std",
    "raw_tissue_mean", "raw_tissue_std", "relative_tissue_mean", "relative_tissue_std",
    "computed_from_split", "train_subject_foot_count",
}
if set(normalization_stats.keys()) != expected_keys:
    issue = f"SECTION 2c: normalization_stats.json keys {set(normalization_stats.keys())} != expected {expected_keys}"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 2c: Save normalization_stats.json ===
Saved: data\processed\normalization_stats.json

{
  "node_feature_names": [
    "mean_abs_temp",
    "std_abs_temp",
    "mean_rel_temp",
    "pixel_count",
    "centroid_row_norm",
    "centroid_col_norm"
  ],
  "node_feature_mean": [
    28.86639404296875,
    0.9413005113601685,
    -0.03255420923233032,
    1935.52587890625,
    0.545102059841156,
    0.502554178237915
  ],
  "node_feature_std": [
    2.974191665649414,
    0.4266098439693451,
    0.6697842478752136,
    1209.862548828125,
    0.2408343106508255,
    0.20934735238552094
  ],
  "raw_tissue_mean": 28.908849716186523,
  "raw_tissue_std": 3.137549638748169,
  "relative_tissue_mean": -2.9530093570429017e-07,
  "relative_tissue_std": 1.2453471422195435,
  "computed_from_split": "train",
  "train_subject_foot_count": 232
}


## Section 3 — Dataset Class

`PlantarThermographyDataset` wraps one split's slice of `combined_manifest_df`. It takes the dataframe and the `normalization_stats` dict from Section 2 (fit once on train, passed unchanged to every split — see Section 4) and returns, per sample, both branches of the dual-branch architecture: the 4-node angiosome graph (from the Phase 6 `.npz`) and the pixel images (raw / relative / colormap, from Phases 4–5), plus a tissue mask and the integer label.

A few normalization details worth calling out explicitly:

- `raw_image` / `relative_image` are z-score normalized using the **train-derived** tissue mean/std, applied only to tissue pixels. Background pixels are explicitly reset to exactly `0.0` **after** normalization — z-scoring `0.0` directly would shift background off zero (since `(0 - mean) / std != 0` in general), and `tissue_mask` is what lets the model later distinguish a real (post-normalization) `0.0` reading from background.
- `colormap_image` only gets scaled to `[0, 1]` by `/255.0` — no z-scoring, since it is a rendered RGB visualization, not a physical measurement.
- Node features are z-scored per-column using the same train-derived stats from Section 2a.


In [7]:
print("=== SECTION 3: Dataset Class ===")


class PlantarThermographyDataset(Dataset):
    """One row per subject+foot. Returns both branches of the dual-branch architecture:
    the 4-node angiosome graph (Phase 6) and the pixel images (Phases 4-5), normalized
    using train-derived statistics supplied via `normalization_stats` (never refit here).
    """

    def __init__(self, manifest_df, normalization_stats):
        self.manifest_df = manifest_df.reset_index(drop=True)
        self.stats = normalization_stats

        self._node_feature_mean = np.array(normalization_stats["node_feature_mean"], dtype=np.float32)
        self._node_feature_std = np.array(normalization_stats["node_feature_std"], dtype=np.float32)
        self._raw_tissue_mean = normalization_stats["raw_tissue_mean"]
        self._raw_tissue_std = normalization_stats["raw_tissue_std"]
        self._relative_tissue_mean = normalization_stats["relative_tissue_mean"]
        self._relative_tissue_std = normalization_stats["relative_tissue_std"]

    def __len__(self):
        return len(self.manifest_df)

    def _normalize_image(self, arr, mask, mean, std):
        """Z-score tissue pixels using train-derived mean/std; force background to exactly 0.0
        AFTER normalization, since (0 - mean) / std is not 0 in general."""
        out = np.zeros_like(arr, dtype=np.float32)
        out[mask] = (arr[mask] - mean) / std
        return out

    def __getitem__(self, idx):
        row = self.manifest_df.iloc[idx]

        # --- Graph branch (Phase 6 .npz) ---
        with np.load(row["npz_path"]) as npz:
            node_features_raw = npz["node_features"].astype(np.float32)  # (4, 6)
            edge_index_base = npz["edge_index_base"].astype(np.int64)    # (2, 8)
            edge_attr_base = npz["edge_attr_base"].astype(np.float32)    # (8, 2)
            edge_index_diag = npz["edge_index_diag"].astype(np.int64)    # (2, 12)
            edge_attr_diag = npz["edge_attr_diag"].astype(np.float32)    # (12, 2)

        node_features = (node_features_raw - self._node_feature_mean) / self._node_feature_std

        # --- Image branch (Phases 4-5) ---
        mask = np.load(row["mask_path"])                     # boolean, (208, 96)
        raw_arr = np.load(row["raw_array_path"])              # float32, (208, 96)
        rel_arr = np.load(row["relative_array_path"])          # float32, (208, 96)
        colormap_arr = np.array(Image.open(row["colormap_png_path"]))  # uint8, (208, 96, 3)

        raw_norm = self._normalize_image(raw_arr, mask, self._raw_tissue_mean, self._raw_tissue_std)
        rel_norm = self._normalize_image(rel_arr, mask, self._relative_tissue_mean, self._relative_tissue_std)
        colormap_norm = (colormap_arr.astype(np.float32) / 255.0).transpose(2, 0, 1)  # (3, 208, 96)

        sample = {
            "node_features": torch.from_numpy(node_features.astype(np.float32)),
            "edge_index_base": torch.from_numpy(edge_index_base),
            "edge_attr_base": torch.from_numpy(edge_attr_base),
            "edge_index_diag": torch.from_numpy(edge_index_diag),
            "edge_attr_diag": torch.from_numpy(edge_attr_diag),
            "raw_image": torch.from_numpy(raw_norm).unsqueeze(0),           # (1, 208, 96)
            "relative_image": torch.from_numpy(rel_norm).unsqueeze(0),      # (1, 208, 96)
            "colormap_image": torch.from_numpy(colormap_norm),              # (3, 208, 96)
            "tissue_mask": torch.from_numpy(mask.astype(np.float32)).unsqueeze(0),  # (1, 208, 96)
            "label": torch.tensor(1 if row["group"] == "DM" else 0, dtype=torch.int64),
            "subject_id": str(row["subject_id"]),
            "foot_side": str(row["foot_side"]),
            "gender": str(row["gender"]),
            "group": str(row["group"]),
            "split": str(row["split"]),
        }
        return sample


print("PlantarThermographyDataset defined.")


=== SECTION 3: Dataset Class ===
PlantarThermographyDataset defined.


## Section 4 — Build the Three Datasets

Filter `combined_manifest_df` by `split` and construct `train_dataset`, `val_dataset`, `test_dataset`, each passing the **same** `normalization_stats` dict fit once on train in Section 2 — never refit per split, which would reintroduce the leakage Section 2 was built to avoid.


In [8]:
print("=== SECTION 4: Building the Three Datasets ===")

val_manifest_df = combined_manifest_df.loc[combined_manifest_df["split"] == "val"].reset_index(drop=True)
test_manifest_df = combined_manifest_df.loc[combined_manifest_df["split"] == "test"].reset_index(drop=True)
# train_df already built in Section 2a from combined_manifest_df; reuse it here for consistency.

train_dataset = PlantarThermographyDataset(train_df, normalization_stats)
val_dataset = PlantarThermographyDataset(val_manifest_df, normalization_stats)
test_dataset = PlantarThermographyDataset(test_manifest_df, normalization_stats)

print(f"len(train_dataset): {len(train_dataset)}")
print(f"len(val_dataset):   {len(val_dataset)}")
print(f"len(test_dataset):  {len(test_dataset)}")
print(f"Sum: {len(train_dataset) + len(val_dataset) + len(test_dataset)} (expected 334)")

if len(train_dataset) + len(val_dataset) + len(test_dataset) != 334:
    issue = "SECTION 4: train + val + test dataset sizes do not sum to 334"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 4: Building the Three Datasets ===
len(train_dataset): 232
len(val_dataset):   50
len(test_dataset):  52
Sum: 334 (expected 334)


## Section 5 — Subject-Level Leakage Re-Check

Extract the set of unique `subject_id`s underlying each of the three `Dataset` objects' dataframes and confirm they are pairwise disjoint. This should already be guaranteed by Phase 5's own leakage check, but the `DataLoader` is the last chance to catch a bug before training, so it's re-verified here at the `Dataset`-object level rather than just trusted.


In [9]:
print("=== SECTION 5: Subject-Level Leakage Re-Check ===")

train_subject_ids = set(train_dataset.manifest_df["subject_id"])
val_subject_ids = set(val_dataset.manifest_df["subject_id"])
test_subject_ids = set(test_dataset.manifest_df["subject_id"])

overlap_train_val = train_subject_ids & val_subject_ids
overlap_train_test = train_subject_ids & test_subject_ids
overlap_val_test = val_subject_ids & test_subject_ids

pairwise_disjoint = not (overlap_train_val or overlap_train_test or overlap_val_test)

print(f"Unique subjects -- train: {len(train_subject_ids)}, val: {len(val_subject_ids)}, test: {len(test_subject_ids)}")
print(f"train & val overlap:  {len(overlap_train_val)}")
print(f"train & test overlap: {len(overlap_train_test)}")
print(f"val & test overlap:   {len(overlap_val_test)}")
print()
print(f"Subject-level leakage re-check: {'PASSED' if pairwise_disjoint else 'FAILED'}")

if not pairwise_disjoint:
    issue = "SECTION 5: train/val/test Dataset objects share at least one subject_id"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 5: Subject-Level Leakage Re-Check ===
Unique subjects -- train: 116, val: 25, test: 26
train & val overlap:  0
train & test overlap: 0
val & test overlap:   0

Subject-level leakage re-check: PASSED


## Section 6 — DataLoaders

`train_loader` (`batch_size=16, shuffle=True`), `val_loader` and `test_loader` (`batch_size=16, shuffle=False`), all using PyTorch's default `collate_fn`.

**Why no custom `collate_fn` is needed here:** every sample returned by `PlantarThermographyDataset.__getitem__` has an identical tensor shape for every key, regardless of which subject+foot it came from — Phase 4 padded every array to the same canonical `(208, 96)` shape, and Phase 6 gave every graph the same fixed 4-node / 8-or-12-edge structure. The default collator stacks same-shaped tensors along a new batch dimension automatically, and the string metadata fields (`subject_id`, `foot_side`, `gender`, `group`, `split`) are collated into plain lists of strings with no special handling required. A custom `collate_fn` only earns its complexity when samples vary in shape (e.g. variable-length sequences or the *original*, pre-padding raw CSV grids from Phase 1) — that isn't the case anywhere in this pipeline past Phase 4.


In [10]:
print("=== SECTION 6: DataLoaders ===")

BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"train_loader: {len(train_loader)} batches over {len(train_dataset)} samples (batch_size={BATCH_SIZE}, shuffle=True)")
print(f"val_loader:   {len(val_loader)} batches over {len(val_dataset)} samples (batch_size={BATCH_SIZE}, shuffle=False)")
print(f"test_loader:  {len(test_loader)} batches over {len(test_dataset)} samples (batch_size={BATCH_SIZE}, shuffle=False)")


=== SECTION 6: DataLoaders ===
train_loader: 15 batches over 232 samples (batch_size=16, shuffle=True)
val_loader:   4 batches over 50 samples (batch_size=16, shuffle=False)
test_loader:  4 batches over 52 samples (batch_size=16, shuffle=False)


## Section 7 — Batch Sanity Check

Pull the first batch from each of the three loaders and print the shape/dtype of every tensor key, confirming `label` contains only `0`s and `1`s. Separately, confirm the *last* batch of each loader has size `min(BATCH_SIZE, len(dataset) - BATCH_SIZE * (num_batches - 1))` — i.e. `shuffle=True`/`False` and no `drop_last` mean the final batch of an epoch is only as large as whatever's left over, and this is checked by actually iterating each loader to its last batch rather than assumed from the dataset size alone. Also inspect a single (non-batched) sample from `train_dataset` directly: `raw_image` min/max/mean should sit roughly centered near 0 for tissue pixels (z-score normalized), and `colormap_image` min/max should fall within `[0, 1]`.


In [11]:
print("=== SECTION 7: Batch Sanity Check ===")

TENSOR_KEYS = [
    "node_features", "edge_index_base", "edge_attr_base", "edge_index_diag", "edge_attr_diag",
    "raw_image", "relative_image", "colormap_image", "tissue_mask", "label",
]

batch_check_passed = True

for loader_name, loader, dataset in [
    ("train_loader", train_loader, train_dataset),
    ("val_loader", val_loader, val_dataset),
    ("test_loader", test_loader, test_dataset),
]:
    print(f"--- {loader_name} ---")
    batch = next(iter(loader))

    for key in TENSOR_KEYS:
        t = batch[key]
        print(f"  {key:16s} shape={tuple(t.shape)}  dtype={t.dtype}")

    labels_ok = bool(((batch["label"] == 0) | (batch["label"] == 1)).all())
    print(f"  label contains only 0/1: {labels_ok}")
    if not labels_ok:
        batch_check_passed = False
        issue = f"SECTION 7: {loader_name} batch['label'] contains values other than 0/1"
        print(f"!!! {issue}")
        issues_log.append(issue)

    # First-batch size is only informative when len(dataset) < BATCH_SIZE (dataset smaller
    # than one batch); otherwise the first batch is always full. The genuine partial-batch
    # case -- the *last* batch of an epoch -- is checked separately just below by actually
    # iterating the loader to its end, since shuffle=True means which samples land in the
    # last batch differs, but its *size* is deterministic (drop_last defaults to False).
    first_expected = min(BATCH_SIZE, len(dataset))
    first_actual = batch["label"].shape[0]
    first_size_ok = first_actual == first_expected
    print(f"  first batch size: {first_actual} (expected min(BATCH_SIZE, len(dataset)) = {first_expected}): "
          f"{'OK' if first_size_ok else 'MISMATCH'}")
    if not first_size_ok:
        batch_check_passed = False
        issue = f"SECTION 7: {loader_name} first batch size {first_actual} != expected {first_expected}"
        print(f"!!! {issue}")
        issues_log.append(issue)

    last_batch = None
    for last_batch in loader:
        pass
    last_actual = last_batch["label"].shape[0]
    n_batches = len(loader)
    last_expected = len(dataset) - BATCH_SIZE * (n_batches - 1)
    last_size_ok = last_actual == last_expected
    print(f"  last batch size:  {last_actual} (expected len(dataset) - BATCH_SIZE*(num_batches-1) = {last_expected}): "
          f"{'OK' if last_size_ok else 'MISMATCH'}")
    if not last_size_ok:
        batch_check_passed = False
        issue = f"SECTION 7: {loader_name} last batch size {last_actual} != expected {last_expected}"
        print(f"!!! {issue}")
        issues_log.append(issue)
    print()

print(f"Batch sanity check (shapes/dtypes/labels/sizes across all 3 loaders): {'PASSED' if batch_check_passed else 'FAILED'}")


=== SECTION 7: Batch Sanity Check ===
--- train_loader ---
  node_features    shape=(16, 4, 6)  dtype=torch.float32
  edge_index_base  shape=(16, 2, 8)  dtype=torch.int64
  edge_attr_base   shape=(16, 8, 2)  dtype=torch.float32
  edge_index_diag  shape=(16, 2, 12)  dtype=torch.int64
  edge_attr_diag   shape=(16, 12, 2)  dtype=torch.float32
  raw_image        shape=(16, 1, 208, 96)  dtype=torch.float32
  relative_image   shape=(16, 1, 208, 96)  dtype=torch.float32
  colormap_image   shape=(16, 3, 208, 96)  dtype=torch.float32
  tissue_mask      shape=(16, 1, 208, 96)  dtype=torch.float32
  label            shape=(16,)  dtype=torch.int64
  label contains only 0/1: True
  first batch size: 16 (expected min(BATCH_SIZE, len(dataset)) = 16): OK
  last batch size:  8 (expected len(dataset) - BATCH_SIZE*(num_batches-1) = 8): OK

--- val_loader ---
  node_features    shape=(16, 4, 6)  dtype=torch.float32
  edge_index_base  shape=(16, 2, 8)  dtype=torch.int64
  edge_attr_base   shape=(16, 8, 2) 

In [12]:
print("=== SECTION 7 (cont.): Single-Sample Inspection ===")

sample = train_dataset[0]

raw_image = sample["raw_image"]
colormap_image = sample["colormap_image"]

print(f"train_dataset[0] subject_id={sample['subject_id']}, foot_side={sample['foot_side']}, "
      f"group={sample['group']}, label={int(sample['label'])}")
print()
print(f"raw_image       min={raw_image.min().item(): .4f}  max={raw_image.max().item(): .4f}  "
      f"mean={raw_image.mean().item(): .4f}")
print(f"colormap_image  min={colormap_image.min().item(): .4f}  max={colormap_image.max().item(): .4f}")

colormap_in_range = bool((colormap_image.min() >= 0.0) and (colormap_image.max() <= 1.0))
print(f"colormap_image within [0, 1]: {colormap_in_range}")
if not colormap_in_range:
    issue = "SECTION 7: colormap_image for train_dataset[0] falls outside [0, 1]"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 7 (cont.): Single-Sample Inspection ===
train_dataset[0] subject_id=CG003, foot_side=L, group=Control, label=0

raw_image       min=-1.5798  max= 1.1548  mean= 0.0845
colormap_image  min= 0.0000  max= 1.0000
colormap_image within [0, 1]: True


## Section 8 — Label Cross-Check

For 5 random samples (`RANDOM_SEED = 42`, consistent with prior notebooks) drawn from `train_dataset`, print `subject_id`, `foot_side`, the dataset's `label` tensor value, and the `group` string, and confirm `label == 1` exactly when `group == "DM"` and `label == 0` exactly when `group == "Control"`.


In [13]:
print("=== SECTION 8: Label Cross-Check ===")

rng = np.random.default_rng(RANDOM_SEED)
sample_indices = rng.choice(len(train_dataset), size=5, replace=False)

label_check_passed = True
for idx in sample_indices:
    s = train_dataset[int(idx)]
    label = int(s["label"])
    group = s["group"]
    expected_label = 1 if group == "DM" else 0
    ok = label == expected_label
    label_check_passed &= ok
    print(f"  idx={idx:4d}  subject_id={s['subject_id']:8s}  foot_side={s['foot_side']}  "
          f"label={label}  group={group:8s}  {'OK' if ok else 'MISMATCH'}")

print()
print(f"Label cross-check: {'PASSED' if label_check_passed else 'FAILED'}")
if not label_check_passed:
    issue = "SECTION 8: at least one sampled label did not match its group (DM=1 / Control=0)"
    print(f"!!! {issue}")
    issues_log.append(issue)


=== SECTION 8: Label Cross-Check ===
  idx= 177  subject_id=DM080     foot_side=R  label=1  group=DM        OK
  idx= 101  subject_id=DM030     foot_side=R  label=1  group=DM        OK
  idx= 150  subject_id=DM062     foot_side=L  label=1  group=DM        OK
  idx=  20  subject_id=CG016     foot_side=L  label=0  group=Control   OK
  idx= 100  subject_id=DM030     foot_side=L  label=1  group=DM        OK

Label cross-check: PASSED


## Section 9 — Final Summary


In [14]:
print("=== SECTION 9: Final Summary ===")

print("Dataset sizes:")
print(f"  train: {len(train_dataset)}")
print(f"  val:   {len(val_dataset)}")
print(f"  test:  {len(test_dataset)}")
print()

norm_stats_saved_ok = NORM_STATS_PATH.exists() and set(normalization_stats.keys()) == expected_keys
print(f"normalization_stats.json saved with all expected keys: {norm_stats_saved_ok}")
print()

print(f"Subject-level leakage re-check (Section 5): {'PASSED' if pairwise_disjoint else 'FAILED'}")
print(f"Batch sanity check (Section 7): {'PASSED' if batch_check_passed else 'FAILED'}")
print(f"Label cross-check (Section 8): {'PASSED' if label_check_passed else 'FAILED'}")
print()

if issues_log:
    print(f"{len(issues_log)} issue(s) found:")
    for issue in issues_log:
        print(f"  - {issue}")
else:
    print("0 issues found.")


=== SECTION 9: Final Summary ===
Dataset sizes:
  train: 232
  val:   50
  test:  52

normalization_stats.json saved with all expected keys: True

Subject-level leakage re-check (Section 5): PASSED
Batch sanity check (Section 7): PASSED
Label cross-check (Section 8): PASSED

0 issues found.
